# 03 — Generalized (shared-parameter) calibration

One shared SteelMPF vector fit jointly over a small train list
(same idea as `optimize_generalized_brb_mse.py`).

Train Names should have `generalized_weight > 0` in
`config/calibration/BRB-Specimens.csv`. Default demo: `PC250`, `PC350`, `PC3SB`.

Edit the **Parameters** cell, then run all. Outputs: `results/notebooks/generalized/`.


## Parameters (edit me)


In [ ]:
from pathlib import Path
import sys

# --- edit these ---
TRAIN_SPECIMENS = ["PC250", "PC350", "PC3SB"]  # keep small for a notebook demo
PREPARE_DATA = True

SET_ID_ROW = {
    "set_id": 1,
    "inherit_from_set": -999,
    "E": 29000,
    "b_p": 0.005,
    "b_n": 0.025,
    "R0": 20,
    "cR1": 0.8875,
    "cR2": 0.15,
    "a1": 0.04,
    "a2": 1.0,
    "a3": 0.04,
    "a4": 1.0,
    "optimize_params": ["b_p", "b_n", "cR1", "cR2", "a1", "a3"],
    "w_feat_l2": 1,
    "w_feat_l1": 0,
    "w_energy_l2": 0,
    "w_energy_l1": 0,
    "w_unordered_binenv_l2": 0,
    "w_unordered_binenv_l1": 0,
    "use_amplitude_weights": True,
    "amplitude_weight_power": 2,
    "amplitude_weight_eps": 0.05,
}
# ------------------

ROOT = Path.cwd()
if not (ROOT / "scripts" / "examples" / "notebook_support.py").is_file():
    raise SystemExit("Run this notebook from the BRB-Calibration repo root.")
sys.path.insert(0, str(ROOT / "scripts" / "examples"))
from notebook_support import display_png, run_generalized_demo  # noqa: E402
print("train:", TRAIN_SPECIMENS)


## Run generalized optimize + overlays


In [ ]:
paths = run_generalized_demo(
    TRAIN_SPECIMENS,
    set_id_row=SET_ID_ROW,
    prepare_data=PREPARE_DATA,
)
print("params: ", paths["params"])
print("metrics:", paths["metrics"])
print("plots:  ", paths["plots_dir"])


## Results


In [ ]:
import pandas as pd

params = pd.read_csv(paths["params"])
metrics = pd.read_csv(paths["metrics"])
display(params.head(12))
cols = ["Name", "set_id", "final_J_total", "final_J_feat_raw"]
display(metrics[cols].head(12) if set(cols).issubset(metrics.columns) else metrics.head(12))

sid = int(SET_ID_ROW["set_id"])
combined = paths["plots_dir"] / f"config_set_{sid}" / f"set{sid}_combined_force_def_norm.png"
if combined.is_file():
    display_png(combined, width=900)
else:
    for name in TRAIN_SPECIMENS:
        cands = list(paths["plots_dir"].rglob(f"*{name}*force_def_norm.png"))
        if cands:
            print(name, "->", cands[0])
            display_png(cands[0], width=520)
        else:
            print(name, ": no overlay found under", paths["plots_dir"])
